In [ ]:
# Se importan las herramientas para representar intereses y construir segmentos reproducibles.

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:
# Se cargan perfiles de estudiantes y sus menciones de intereses.

data = pd.read_csv("../data/snsdata.csv")
data.head()

In [ ]:
# ¿Qué edades requieren corrección antes de caracterizar los grupos?

data["age_clean"] = data["age"].where(data["age"].between(13, 20, inclusive="left"))
age_medians = data.groupby("gradyear")["age_clean"].median()
data["age_imputed"] = data["age_clean"].fillna(data["gradyear"].map(age_medians))
data[["gradyear", "age", "age_imputed", "friends"]].describe()

In [ ]:
# Se separan intereses usados para agrupar de atributos usados solo para caracterizar.

interest_columns = data.columns[4:40].tolist()
profile_columns = ["gradyear", "gender", "age_imputed", "friends"]
len(interest_columns), profile_columns

In [ ]:
# Se ponderan intereses y se asignan cinco grupos con una semilla reproducible.

tfidf = TfidfTransformer()
interest_matrix = tfidf.fit_transform(data[interest_columns])
kmeans = KMeans(n_clusters=5, n_init=30, random_state=42)
data["cluster"] = kmeans.fit_predict(interest_matrix)
cluster_sizes = data["cluster"].value_counts().sort_index().rename("n").to_frame()
cluster_sizes["percentage"] = (100 * cluster_sizes["n"] / len(data)).round(2)
cluster_sizes

In [ ]:
# ¿Qué intereses distinguen a cada grupo respecto de los demás?

centers = pd.DataFrame(kmeans.cluster_centers_, columns=interest_columns)
top_interests = []
for cluster_id in centers.index:
    for interest, score in centers.loc[cluster_id].nlargest(6).items():
        top_interests.append({"cluster": cluster_id, "interest": interest, "score": score})
top_interests = pd.DataFrame(top_interests)
top_interests.sort_values(["cluster", "score"], ascending=[True, False])

In [ ]:
# Se calculan perfiles complementarios sin usar edad ni género para construir los segmentos.

data["gender_profile"] = data["gender"].fillna("Unknown")
cluster_profiles = data.groupby("cluster").agg(n=("cluster", "size"), age_mean=("age_imputed", "mean"), friends_mean=("friends", "mean")).round(2)
gender_profiles = pd.crosstab(data["cluster"], data["gender_profile"], normalize="index").mul(100).round(1)
gradyear_profiles = pd.crosstab(data["cluster"], data["gradyear"], normalize="index").mul(100).round(1)
cluster_profiles.join(cluster_sizes[["percentage"]])

In [ ]:
# Se conservan las evidencias para que la interpretación de los grupos ocurra durante la clase.

from pathlib import Path

submission_dir = Path("../submission")
data.to_csv(submission_dir / "segmented.csv", index=False)
cluster_sizes.reset_index().to_csv(submission_dir / "cluster_sizes.csv", index=False)
top_interests.to_csv(submission_dir / "top_interests.csv", index=False)
cluster_profiles.reset_index().to_csv(submission_dir / "cluster_profiles.csv", index=False)
gender_profiles.reset_index().to_csv(submission_dir / "gender_profiles.csv", index=False)
gradyear_profiles.reset_index().to_csv(submission_dir / "gradyear_profiles.csv", index=False)